<a href="https://colab.research.google.com/github/kapitalPsol/tf-models-savedmodels/blob/main/TemperatureDataPulls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Pulls daily min/max temperature data from NOAA's public Access Data Service
(no login, no key required) and computes average days/year below several
cold thresholds -- for use in the PTC cold-start verification workbook.

Works as-is in Google Colab or any local Jupyter environment with internet
access. Both `requests` and `pandas` are preinstalled in Colab; if running
locally and either is missing, run:  pip install requests pandas
"""

import requests
import pandas as pd
from io import StringIO

def get_threshold_days(station_id, start="1991-01-01", end="2020-12-31"):
    """
    Fetches daily TMIN/TMAX (in whole degrees F) for a GHCN-Daily station
    and returns:
      - a DataFrame of yearly counts of days <= each threshold
      - a summary Series of the multi-year average for each threshold
    """
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": start,
        "endDate": end,
        "dataTypes": "TMIN,TMAX",
        "format": "csv",
        "units": "standard",  # returns Fahrenheit directly
    }

    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()

    df = pd.read_csv(StringIO(resp.text))
    df["DATE"] = pd.to_datetime(df["DATE"])
    df["YEAR"] = df["DATE"].dt.year
    df["TMIN"] = pd.to_numeric(df["TMIN"], errors="coerce")
    df["TMAX"] = pd.to_numeric(df["TMAX"], errors="coerce")

    thresholds = {
        "days_TMAX_le_32F": ("TMAX", 32),
        "days_TMIN_le_0F": ("TMIN", 0),
        "days_TMIN_le_-20F": ("TMIN", -20),
        "days_TMIN_le_-29F": ("TMIN", -29),
        "days_TMIN_le_-40F": ("TMIN", -40),
    }

    yearly = pd.DataFrame(index=sorted(df["YEAR"].unique()))
    for label, (col, thresh) in thresholds.items():
        yearly[label] = (
            df[df[col] <= thresh].groupby("YEAR").size()
        )
    yearly = yearly.fillna(0)

    summary = yearly.mean().round(1)
    return yearly, summary


if __name__ == "__main__":
    # Confirmed station ID:
    HOUGHTON_MI = "USW00014858"

    # Look up Bemidji's exact ID first at:
    #   https://www.ncei.noaa.gov/cdo-web/search
    # then paste it in below.
    BEMIDJI_MN = "USC00210643"

    stations = {
        "Houghton, MI (49931)": HOUGHTON_MI,
        "Bemidji, MN (56601)": BEMIDJI_MN,
        "Butte, MT (59701)": "USW00024135",
        "Anchorage, AK (99502)": "USW00026451",
    }

    for label, sid in stations.items():
        print(f"\n=== {label} (station {sid}) ===")
        if sid == "PUT_STATION_ID_HERE":
            print("  Skipped -- look up the real station id first (see comment above).")
            continue
        try:
            yearly, summary = get_threshold_days(sid)
            print(f"  Years of data: {len(yearly)}")
            print("  Average days/year:")
            for k, v in summary.items():
                print(f"    {k}: {v}")
        except Exception as e:
            print(f"  Error fetching data: {e}")


=== Houghton, MI (49931) (station USW00014858) ===
  Years of data: 30
  Average days/year:
    days_TMAX_le_32F: 95.9
    days_TMIN_le_0F: 16.7
    days_TMIN_le_-20F: 0.6
    days_TMIN_le_-29F: 0.0
    days_TMIN_le_-40F: 0.0

=== Bemidji, MN (56601) (station USC00210643) ===
  Years of data: 20
  Average days/year:
    days_TMAX_le_32F: 33.2
    days_TMIN_le_0F: 18.8
    days_TMIN_le_-20F: 5.6
    days_TMIN_le_-29F: 2.0
    days_TMIN_le_-40F: 0.2

=== Butte, MT (59701) (station USW00024135) ===
  Years of data: 30
  Average days/year:
    days_TMAX_le_32F: 52.0
    days_TMIN_le_0F: 30.0
    days_TMIN_le_-20F: 3.8
    days_TMIN_le_-29F: 0.8
    days_TMIN_le_-40F: 0.0

=== Anchorage, AK (99502) (station USW00026451) ===
  Years of data: 30
  Average days/year:
    days_TMAX_le_32F: 104.5
    days_TMIN_le_0F: 18.4
    days_TMIN_le_-20F: 0.3
    days_TMIN_le_-29F: 0.0
    days_TMIN_le_-40F: 0.0


In [8]:
"""
Pulls daily min/max temperature data from NOAA's public Access Data Service
(no login, no key required) and computes average days/year below several
cold thresholds -- for use in the Hot ambient verification workbook.

Works as-is in Google Colab or any local Jupyter environment with internet
access. Both `requests` and `pandas` are preinstalled in Colab; if running
locally and either is missing, run:  pip install requests pandas
"""

import requests
import pandas as pd
from io import StringIO

def get_threshold_days(station_id, start="1991-01-01", end="2020-12-31"):
    """
    Fetches daily TMIN/TMAX (in whole degrees F) for a GHCN-Daily station
    and returns:
      - a DataFrame of yearly counts of days <= each threshold
      - a summary Series of the multi-year average for each threshold
    """
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": start,
        "endDate": end,
        "dataTypes": "TMIN,TMAX",
        "format": "csv",
        "units": "standard",  # returns Fahrenheit directly
    }

    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()

    df = pd.read_csv(StringIO(resp.text))
    df["DATE"] = pd.to_datetime(df["DATE"])
    df["YEAR"] = df["DATE"].dt.year
    df["TMIN"] = pd.to_numeric(df["TMIN"], errors="coerce")
    df["TMAX"] = pd.to_numeric(df["TMAX"], errors="coerce")

    thresholds = {
        "days_TMIN_ge_90F": ("TMIN", 90),
        "days_TMAX_ge_109F": ("TMAX", 109),
        "days_TMAX_ge_120F": ("TMAX", 120),
        "days_TMAX_ge_125": ("TMAX", 125),
        "days_TMAX_ge_130F": ("TMAX", 130),
    }

    yearly = pd.DataFrame(index=sorted(df["YEAR"].unique()))
    for label, (col, thresh) in thresholds.items():
        yearly[label] = (
            df[df[col] >= thresh].groupby("YEAR").size()
        )
    yearly = yearly.fillna(0)

    summary = yearly.mean().round(1)
    return yearly, summary


if __name__ == "__main__":
    # Confirmed station ID:
    LasVegas_NV = "USW00023169"
    DeathValley_CA = "USC00042319"
    Dallas_TX = "USW00093125"
    Houston_TX = "USW00012960"
    Phoenix_AZ = "USW00023174"
    Miami_FL = "USW00012839"

    stations = {
        "Las Vegas, NV": LasVegas_NV,
        "Death Valley, CA": DeathValley_CA,
        "Dallas, TX": Dallas_TX,
        "Houston, TX": Houston_TX,
        "Phoenix, AZ": Phoenix_AZ,
        "Miami, FL": Miami_FL,

    }

    for label, sid in stations.items():
        print(f"\n=== {label} (station {sid}) ===")
        if sid == "PUT_STATION_ID_HERE":
            print("  Skipped -- look up the real station id first (see comment above).")
            continue
        try:
            yearly, summary = get_threshold_days(sid)
            print(f"  Years of data: {len(yearly)}")
            print("  Average days/year:")
            for k, v in summary.items():
                print(f"    {k}: {v}")
        except Exception as e:
            print(f"  Error fetching data: {e}")


=== Las Vegas, NV (station USW00023169) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_90F: 1.7
    days_TMAX_ge_109F: 14.3
    days_TMAX_ge_120F: 0.0
    days_TMAX_ge_125: 0.0
    days_TMAX_ge_130F: 0.0

=== Death Valley, CA (station USC00042319) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_90F: 31.5
    days_TMAX_ge_109F: 102.5
    days_TMAX_ge_120F: 23.1
    days_TMAX_ge_125: 4.2
    days_TMAX_ge_130F: 0.0

=== Dallas, TX (station USW00093125) ===
  Years of data: 0
  Average days/year:
    days_TMIN_ge_90F: nan
    days_TMAX_ge_109F: nan
    days_TMAX_ge_120F: nan
    days_TMAX_ge_125: nan
    days_TMAX_ge_130F: nan

=== Houston, TX (station USW00012960) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_90F: 0.0
    days_TMAX_ge_109F: 0.1
    days_TMAX_ge_120F: 0.0
    days_TMAX_ge_125: 0.0
    days_TMAX_ge_130F: 0.0

=== Phoenix, AZ (station USW00023174) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_90F: 0.0
    days_TMA

In [ ]:
"""
Pulls daily min/max temperature data from NOAA's public Access Data Service
(no login, no key required) and computes average days/year below several
cold thresholds -- for use in the Hot ambient verification workbook.

Works as-is in Google Colab or any local Jupyter environment with internet
access. Both `requests` and `pandas` are preinstalled in Colab; if running
locally and either is missing, run:  pip install requests pandas
"""

import requests
import pandas as pd
from io import StringIO

def get_threshold_days(station_id, start="1991-01-01", end="2020-12-31"):
    """
    Fetches daily TMIN/TMAX (in whole degrees F) for a GHCN-Daily station
    and returns:
      - a DataFrame of yearly counts of days <= each threshold
      - a summary Series of the multi-year average for each threshold
    """
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": start,
        "endDate": end,
        "dataTypes": "TMIN,TMAX",
        "format": "csv",
        "units": "standard",  # returns Fahrenheit directly
    }

    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()

    df = pd.read_csv(StringIO(resp.text))
    df["DATE"] = pd.to_datetime(df["DATE"])
    df["YEAR"] = df["DATE"].dt.year
    df["TMIN"] = pd.to_numeric(df["TMIN"], errors="coerce")
    df["TMAX"] = pd.to_numeric(df["TMAX"], errors="coerce")

    thresholds = {
        "days_TMIN_ge_90F": ("TMIN", 90),
        "days_TMAX_ge_109F": ("TMAX", 109),
        "days_TMAX_ge_120F": ("TMAX", 120),
        "days_TMAX_ge_125": ("TMAX", 125),
        "days_TMAX_ge_130F": ("TMAX", 130),
    }

    yearly = pd.DataFrame(index=sorted(df["YEAR"].unique()))
    for label, (col, thresh) in thresholds.items():
        yearly[label] = (
            df[df[col] >= thresh].groupby("YEAR").size()
        )
    yearly = yearly.fillna(0)

    summary = yearly.mean().round(1)
    return yearly, summary


if __name__ == "__main__":
    # Confirmed station ID:
    LasVegas_NV = "USW00023169"

    # Look up Bemidji's exact ID first at:
    #   https://www.ncei.noaa.gov/cdo-web/search
    # then paste it in below.
    DeathValley_CA = "USC00042319"

    stations = {
        "Houghton, MI (49931)": HOUGHTON_MI,
        "Bemidji, MN (56601)": BEMIDJI_MN,
        "Butte, MT (59701)": "USW00024135",
        "Anchorage, AK (99502)": "USW00026451",
    }

    for label, sid in stations.items():
        print(f"\n=== {label} (station {sid}) ===")
        if sid == "PUT_STATION_ID_HERE":
            print("  Skipped -- look up the real station id first (see comment above).")
            continue
        try:
            yearly, summary = get_threshold_days(sid)
            print(f"  Years of data: {len(yearly)}")
            print("  Average days/year:")
            for k, v in summary.items():
                print(f"    {k}: {v}")
        except Exception as e:
            print(f"  Error fetching data: {e}")


=== Houghton, MI (49931) (station USW00014858) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_50F: 84.2
    days_TMAX_ge_109F: 0.0
    days_TMAX_ge_120F: 0.0
    days_TMAX_ge_125: 0.0
    days_TMAX_ge_130F: 0.0

=== Bemidji, MN (56601) (station USC00210643) ===
  Years of data: 20
  Average days/year:
    days_TMIN_ge_50F: 23.2
    days_TMAX_ge_109F: 0.0
    days_TMAX_ge_120F: 0.0
    days_TMAX_ge_125: 0.0
    days_TMAX_ge_130F: 0.0

=== Butte, MT (59701) (station USW00024135) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_50F: 14.4
    days_TMAX_ge_109F: 0.0
    days_TMAX_ge_120F: 0.0
    days_TMAX_ge_125: 0.0
    days_TMAX_ge_130F: 0.0

=== Anchorage, AK (99502) (station USW00026451) ===
  Years of data: 30
  Average days/year:
    days_TMIN_ge_50F: 64.8
    days_TMAX_ge_109F: 0.0
    days_TMAX_ge_120F: 0.0
    days_TMAX_ge_125: 0.0
    days_TMAX_ge_130F: 0.0


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


